# Informed Search
## Path Finding Search with A*
### Introduction
Suppose we want to find the shortest path in a grid with many obstacles.

The world will be simple. It will be a rectangular grid of dots `.` representing empty spaces, and walls represented by hashes `#`. The start space is $(0,0)$ (the top left space), and the goal is the bottom right space: $(\text{height}-1, \text{width}-1)$.

In [6]:
import random
import mazegen

random.seed(0)

height = 10
width = 20

maze = mazegen.mazegen(height, width)
mazegen.print_maze(maze)

..........#.....####
..#.....##.#.#....#.
..#..#...##....#....
#.#..............###
##....###.#...##....
.#......#.....#.....
...#..#...##........
...#...##..#..#.....
......#........#...#
............#.......


We can query any position of the board. In other words, you could think of this as path finding for an agent with perfect information about the world: it does not need physically act or move within the space to explore routes to discover where they lead. You might want to imagine possible modifications to the algorithm for an agent which can only observe the spaces around it.

### The Heuristic
Informed search uses some evaluation function to estimate which option is best. The function will represent the *cost* of each possible node, so we want to take the lowest one. In our case, a good cost function would be the path length from this space to the goal, and we obviously prefer the lower option.

If we had a perfect cost function we would not need to search at all, we could just follow the path indicated by the known shortest path at each space. Of course we do not, and this is why we need a *heuristic* function, which estimates this cost.

For our example a good heuristic will be the number of squares between the given space and the goal, ignoring obstacles. This is also called the Manhattan distance: for a given row-column space $(r, c)$ and goal $(g_r, g_c)$, <br> 
$f(r, c) = |g_r - r| + |g_c - c|$.<br>
(Since we know the goal is the very last space, we could drop the absolute value signs, but we'll leave it in the code for generality.)

In [8]:
# here is an implementation of our heuristic function
# note this function will be redefined in the code later
def h_cost(space, goal):
    return abs(goal[0] - space[0]) + abs(goal[1] - space[1])

### Greedy Search
We can implement greedy search as a breadth first search that always explores the spaces with the lowest overall heuristic cost.

This will be very similar to the code in the Tower of Hanoi and Missionaries and Cannibals examples, so refresh your memory if you are unsure. 

We are going to use a priority queue to implement the frontier. This means each state is assigned an additional value, and lower values will be popped from the queue first. Think of each item in the queue as a tuple `(cost, (row, column))`. So if the queue contained `(5, (15, 10))` and `(1, (20, 9))`, the latter would be returned first.

But first we create a Node class which allows us to keep track of a Node's parent. The objects need to be comparable to be able to put them in a priority queue, but we only need to implement `__lt__` (less than) for it to work.

In [9]:
class Node:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent

    def __eq__(self, other):
        return self.state == other.state

    def __lt__(self, other):
        return self.state < other.state

    def __hash__(self):
        return hash(self.state)

    def __str__(self):
        return f"Node space {self.state}"

Here is the code for the Frontier. We use a combination of a priority queue and a set to keep track of the nodes in the frontier, for efficient operations.

Don't worry too much about the fine implementation details, just make sure you understand what each function will do (from its name and parameters). The important things are:
* When a state is pushed its heuristic cost is calculated
 * If its cost is lower than an existing one in the frontier, it is updated
* When a state is popped the lowest (best) cost node is returned

In [13]:
from priorityqueue import PriorityQueue

class Frontier:
    # Note the heuristic function is passed in as a parameter
    # Python borrows some nice features from functional programming
    def __init__(self, heuristic, start_node=None):
        # This is the functional-programming bit — the frontier doesn't know or care whether it's greedy's h or a*'s g + h:
        self.heuristic = heuristic
        # orders waiting nodes by cost so we can pop the cheapest in O(log n),
        # Lets us ask "which waiting cell is cheapest?" quickly:
        self.queue = PriorityQueue()
        #It lets us ask "is this cell already in the frontier?" quickly:
        self.states = set()

        if start_node is not None:
            self.push(start_node)
            
    def push(self, node):
        # Call the stored function on this node to get its cost:
        cost = self.heuristic(node)
        # Keep only the cheapest known route to a cell:
        # get_priority returns math.inf for a cell not yet queued, so a new cell always passes,
        # and a repeat passes only if this route is cheaper than the cost
        # already recorded for that cell.
        if cost < self.queue.get_priority(node):
            self.queue.push(node, priority=cost)
            self.states.add(node.state)
        
    def pop(self):
        # Delegates down: Frontier.pop -> PriorityQueue.pop -> heapq.heappop
        # heappop is what guarantees the lowest-priority item comes back.
        node = self.queue.pop()
        # drop the node's state from the set to keep the two in sync
        self.states.remove(node.state)
        return node
        
    def contains(self, state):
        return state in self.states
    
    def length(self):
        return self.queue.length()

### Note to self: the Frontier keeps two structures

Both hold the same membership — every push writes to both, every pop
removes from both — but each is shaped for a different question.

|  | `self.queue` | `self.states` |
|---|---|---|
| Holds | `Node` objects + priority | bare `(row, col)` tuples |
| Ordered? | by cost | not at all |
| Answers | which cell should I expand next? | is this cell already in the frontier?

We'll use this function to determine whether a space (given as a `(row, col)` tuple) is valid, i.e. whether it is an empty space within the boundaries of the maze. This time we choose to do this before creating the node object.

In [14]:
# a node for an invalid cell would be wasted work — and in some cases actively harmful, so better to do this check before creating the node object. 
def valid_space(maze, space):
    # Parentheses let one expression span several lines (the original used \
    # backslashes for this, but those can't have comments after them):
    return (0 <= space[0] < len(maze)              # row is on the board
            and 0 <= space[1] < len(maze[0])       # column is on the board
            and maze[space[0]][space[1]] == '.')   # and it's clear, not a wall '#'

Finally, the search code itself. The basic structure is not that different from the Tower of Hanoi example. The Frontier is already doing all the work to make this a greedy informed search: it calculates the heuristics and chooses which one to serve up next.

In [15]:

def greedy_search(maze, start=(0, 0), goal=None):
    if goal is None:
        # compute your goal: note len(maze) counts rows and len(maze[0]) counts the columns within the first row:
        goal = (len(maze) - 1, len(maze[0]) - 1)

    # here's our Manhattan distance heurstic, as a lambda expression, this is the line that makes it a greedy search:
    # Remember state is just the (row, col) tuple
    heuristic = lambda node: abs(goal[0] - node.state[0]) + abs(goal[1] - node.state[1])
    frontier = Frontier(heuristic, Node(start)) # start node pushed in here
    # explored is the graph-search record of closed cells:
    explored = set()

    # The start node goes into the frontier and comes straight back out, with nothing happening in between.
    # Why: the while loop needs current_node to already hold something before the first iteration, because the loop condition tests it
    # Also a small edge benefit: if the start cell is the goal, the loop never runs and the start node is returned correctly, with number_explored at
    current_node = frontier.pop()   # and start node immediately pulled back out
    number_explored = 0

    #Keep going until the popped node is the goal:
    while not current_node.state == goal:
        current_state = current_node.state

        number_explored += 1
        explored.add(current_state)
        
        # working out the four neighbouring locations,
        # Why is this needed: when the search expands a node, it has to generate the possible moves itself, and on a 4-connected grid that's these four arithmetic operations:
        right = (current_state[0], current_state[1] + 1)
        left = (current_state[0], current_state[1] - 1)
        down = (current_state[0] + 1, current_state[1])
        up = (current_state[0] - 1, current_state[1])
        
        for space in [right, left, down, up]:
            #valid_space throws out anything off the board or on a wall and space not in explored throws out cells already closed:
            if valid_space(maze, space) \
               and space not in explored: 
                # parent=current_node records that you arrived at this neighbour from the cell being expanded
                node = Node(space, parent=current_node) 
                # frontier.push isn't a blind insert - it computes the cost and checks whether this cell is already queued more cheaply
                frontier.push(node)

        # Frontier empty means every reachable cell is exhausted and the goal was never found
        if frontier.length() == 0:
            return None, number_explored

        # Pick the next cell with the lowest h. This single line is the greedy behaviour
        # Go back to step 1 of the while loop with the new current_node
        current_node = frontier.pop()

    # Loop exited, so this node is the goal:
    return current_node, number_explored

# here is the "main" code, we generate a new maze then try the search
# try changing the random seed to try different mazes (not all are solvable)
height = 10
width = 20

random.seed(0)
maze = mazegen.mazegen(height, width)
# Tuple unpacking splits the two return values of greedy_search
final_node, number_explored = greedy_search(maze)

if final_node is None:
    print("No path exists!\n")
    mazegen.print_maze(maze)
else:
    node = final_node
    steps = 0
    # Walk the parent chain backwards from the goal, marking each cell X and counting:
    while node.parent is not None:
        state = node.state
        maze[state[0]][state[1]] = 'X'
        steps += 1
        node = node.parent

    # Why is this here? The while loop exits when node.parent is None, which
    # is the start cell — so it never gets marked inside the above while loop:
    state = node.state
    maze[state[0]][state[1]] = 'X'
    mazegen.print_maze(maze)
    
    print(f"Total steps on path: {steps}")
    print(f"Total states explored: {number_explored}")

XXXXXXXX..#.....####
..#....X##.#.#....#.
..#..#.XX##....#....
#.#.....XXXXXXXXX###
##....###.#...##XXXX
.#......#.....#....X
...#..#...##.......X
...#...##..#..#...XX
......#........#..X#
............#.....XX
Total steps on path: 30
Total states explored: 32


Greedy informed search is great if we want a fast result. In this example the path to the goal is 30 steps long, and only 32 spaces were explored. However, the route is not optimal.

```
XX###
#XXXX
....X
....X
...XX
#..X#
...XX
```

In the bottom-right hand side of the maze, just before getting to the goal, the path doubles back on itself. This route would have been 2 steps shorter:
```
XX###
#XXX.
...X.
...X.
...X.
#..X#
...XX
```

### A* Search
This is where A* search comes in. A* is guaranteed to find the optimal solution, provided we have an _admissible_ and _consistent heuristic_. Refer to the notes or the textbook for more details, but suffice to say our Manhattan distance will meet both conditions.

The only modification required for A* search is to change the cost function. Rather than just using $h(x,y)$, our estimated cost, we now use $g(x,y) + h(x,y)$ to evaluate each space. $g(x,y)$ is the actual cost up to this point, and as before $h(x,y)$ is the estimated cost from the space to the goal.

This just requires a slight modification to our code. Now each state must keep track of the length of the path from the start space. We'll modify the Node class to keep track of the total path cost, which thankfully is always one greater than the cost of the parent node:

In [16]:
class Node:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent
        # Add path cost:
        if parent is None:
            self.path_cost = 0
        else:
            self.path_cost = parent.path_cost + 1

    def __eq__(self, other):
        return self.state == other.state

    def __lt__(self, other):
        return self.state < other.state

    def __hash__(self):
        return hash(self.state)

    def __str__(self):
        return f"Node space {self.state}"

Now we can modify the search code to become A* simply by changing the heuristic function that is passed into the Frontier.

In [18]:
def a_star_search(maze, start=(0, 0), goal=None):
    if goal is None:
        goal = (len(maze) - 1, len(maze[0]) - 1)

    # This is the key line that turns the greedy search into A*
    heuristic = lambda node: node.path_cost + \
                             abs(goal[0] - node.state[0]) + abs(goal[1] - node.state[1])
    frontier = Frontier(heuristic, Node(start))
    explored = set()

    current_node = frontier.pop()
    number_explored = 0
    
    while not current_node.state == goal:
        current_state = current_node.state

        number_explored += 1
        explored.add(current_state)
        
        # the four neigbouring locations
        right = (current_state[0], current_state[1] + 1)
        left = (current_state[0], current_state[1] - 1)
        down = (current_state[0] + 1, current_state[1])
        up = (current_state[0] - 1, current_state[1])
        
        for space in [right, left, down, up]:
            if valid_space(maze, space) \
               and space not in explored:
                node = Node(space, parent=current_node)
                frontier.push(node)

        if frontier.length() == 0:
            return None, number_explored

        current_node = frontier.pop()
    
    return current_node, number_explored

# The main code is mostly the same as above, just have to call the correct function!
height = 10
width = 20

random.seed(0)
maze = mazegen.mazegen(height, width)
final_node, number_explored = a_star_search(maze)

if final_node is None:
    print("No path exists!\n")
    mazegen.print_maze(maze)
else:
    node = final_node
    steps = 0
    while node.parent is not None:
        state = node.state
        maze[state[0]][state[1]] = 'X'
        steps += 1
        node = node.parent

    state = node.state
    maze[state[0]][state[1]] = 'X'
    mazegen.print_maze(maze)
    
    print(f"Total steps on path: {steps}")
    print(f"Total states explored: {number_explored}")

XXXXXXXX..#.....####
..#....X##.#.#....#.
..#..#.XX##....#....
#.#.....XXXXXXXXX###
##....###.#...##XXX.
.#......#.....#...X.
...#..#...##......X.
...#...##..#..#...X.
......#........#..X#
............#.....XX
Total steps on path: 28
Total states explored: 114


As predicted, the A* algorithm found a better path of length 28 rather than 30. To do so, many more spaces had to be explored, just in case they were going to lead to a better path.

You can play around with both algorithms using the code below. Try changing the random seed to get different mazes. Or you can write your own maze directly into the code, there is a helper function in mazegen which will convert a string into a 2D array of characters as required. Try this example, which nicely demonstrates the difference between the algorithms, then try making your own.

```python
maze = '''\
.......
#.####.
..#....
.##.###
.......'''
maze = mazegen.str2array(maze)
```

In [ ]:
# This helper function makes the code cleaner in the next cell

import copy

def display_results(maze, final_node, number_explored):
    if final_node is None:
        # greedy_search / a_star_search return None when the frontier emptied
        # without reaching the goal, i.e. the maze has no clear path.
        print("No path exists!\n")
        mazegen.print_maze(maze)
    else:
        # Work on a COPY: marking cells with 'X' overwrites the maze in place,
        # so without this the second search would see the first one's path and
        # ('X' isn't '.', so valid_space would read those cells as walls.)
        # deepcopy, not copy(), because maze is a list of lists
        # copy.copy (shallow) makes a new outer list, but it fills it with the same row objects. copy.deepcopy makes new rows too.
        maze = copy.deepcopy(maze)
        node = final_node
        steps = 0
        # Walk the parent chain backwards from the goal, marking each cell:
        while node.parent is not None:
            state = node.state
            maze[state[0]][state[1]] = 'X'
            steps += 1
            node = node.parent

        # The loop stops AT the start node (parent is None), so mark it here.
        # steps isn't incremented: it counts moves, not cells (30 steps = 31 X's).
        state = node.state
        maze[state[0]][state[1]] = 'X'
        mazegen.print_maze(maze)

        print(f"Total steps on path: {steps}")
        print(f"Total states explored: {number_explored}")
        print()

In [20]:
height = 10
width = 50

random.seed(1)
maze = mazegen.mazegen(height, width, obstacle_weight=0.3)

final_node, number_explored = greedy_search(maze)
display_results(maze, final_node, number_explored)

final_node, number_explored = a_star_search(maze)
display_results(maze, final_node, number_explored)

X##....#..#.#..#.##...#.............#...##..###.#.
X.##....#...#....#.....##....##.#..#....#.#####...
X#XXXXXXXXXXXXXX####.#....#....XXXXX#........#...#
XXX#.#...##....XXXX.##.#.##...#X##.X###....XXXXXXX
.##...##...##.#..#X#..#.#.#...#X##.XXX.###.X#....X
#.#..##....#..#...X#.#.##.....XX#....X###..X#....X
....#...#....##...XXXXXXXXXXX#X#..#.#X###.XX#.#.#X
.#####....#...##..#.........XXX.#....XXXXXX#.....X
##.....#.##..#.##.###.....#..#...#.##..##.##.#.##X
#....##......#......#.......###..##....#.#.###...X
Total steps on path: 78
Total states explored: 107

X##....#..#.#..#.##...#.............#...##..###.#.
X.##....#...#....#.....##....##.#..#....#.#####...
X#XXXXXXXXXXXXXX####.#....#....XXXXX#........#...#
XXX#.#...##....XXXX.##.#.##...#X##.X###....XXXXXXX
.##...##...##.#..#X#..#.#.#...#X##.XXX.###.X#....X
#.#..##....#..#...X#.#.##...XXXX#....X###..X#....X
....#...#....##...XXXXXXXXXXX#.#..#.#X###.XX#.#.#X
.#####....#...##..#.............#....XXXXXX#.....X
##.....#.##..#.##.###.....#..#